In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from models.model_utils import Pl_model_wrapper
import numpy as np
import random
import time
import pickle
import os
from data.datamodule import Datamodule # type: ignore
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.loggers import WandbLogger
import pytorch_lightning as pl
import hydra
from omegaconf import DictConfig, OmegaConf

from data.data_preprocess import HeteroAddLaplacianEigenvectorPE, SubSample
from data.dataset import LPDataset, ILPDataset
from torch_geometric.transforms import Compose

# Initialize hydra config
cfg = OmegaConf.load('conf/config.yaml')


if cfg.data.task == 'sub':
    ILP = True
    model_name = 'TripartiteHeteroGNN'
    dataset = ILPDataset(cfg.data.datapath,
                    extra_path=f'{cfg.other.ipm_restarts}restarts_'
                                        f'{cfg.model.params.lappe}lap_'
                                        f'{cfg.other.ipm_steps}steps'
                                        f'{"_upper_" + str(cfg.other.upper) if cfg.other.upper is not None else ""}',
                    upper_bound=cfg.other.upper,
                    rand_starts=cfg.other.ipm_restarts)
else:
    ILP = False
    model_name = 'TripartiteHeteroGNNClean'
    dataset = LPDataset(cfg.data.datapath,
                    extra_path=f'{cfg.other.ipm_restarts}restarts_'
                                        f'{cfg.model.params.lappe}lap_'
                                        f'{cfg.other.ipm_steps}steps'
                                        f'{"_upper_" + str(cfg.other.upper) if cfg.other.upper is not None else ""}',
                    upper_bound=cfg.other.upper,
                    rand_starts=cfg.other.ipm_restarts,
                    pre_transform=Compose([HeteroAddLaplacianEigenvectorPE(k=cfg.model.params.lappe),
                                                    SubSample(cfg.other.ipm_steps)]))

model = Pl_model_wrapper.load_from_checkpoint(cfg.eval.ckpt, 
                                    model_name=model_name,
                                    cfg=cfg,
                                    device=cfg.train.device,
                                    ILP=ILP)

model.eval()

In [8]:
import gzip
with gzip.open(os.path.join(f"{cfg.data.datapath}/raw/", f"instance_0.pkl.gz"), "rb") as file:
    ip_pkgs = pickle.load(file)
A,b,c = ip_pkgs[0]
data = dataset.prepare_example(A,b,c)
if ILP:
    vals = model.validation_diffusion(data)
else:
    vals,_ = model(data)

In [11]:
data.gt_primals.shape

torch.Size([30, 8])

In [9]:
vals.shape

torch.Size([30, 8])